[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/11_sft_grpo_unsloth.ipynb)

# 11 · SFT + GRPO on Qwen3 Base（Unsloth + W&B）

结构参考 Unsloth 官方 Colab：
[Qwen3 (4B) Instruct（SFT）](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-Instruct.ipynb) 和
[Qwen3 (4B) GRPO](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb)：
同一个模型对象开 `fast_inference=True`（内置 vLLM），先 **SFT**（OLMo 3 Dolci 数据 + 选择题格式），再 **GRPO**（可验证奖励）。

在 base / SFT / GRPO 三个时间点各做一次 **GPQA 快速评测**（同一个 vLLM 引擎，只输出聚合数字）。
训练曲线和每个阶段的 GPQA 指标会记录到 **Weights & Biases**（Colab 🔑 Secrets 里放 `WANDB_API_KEY`；没有就自动关闭）。

| 模式 | 用途 | SFT | GRPO | 快速评测 |
|---|---|---|---|---|
| `quick` | 验证流程（T4 也行） | 200 Dolci + 200 格式样本，40 步 | 30 步，回答 ≤512 token | Diamond 前 60 题，≤1024 token |
| `medium` | 1.7B 看变化 | 1000 + 500，150 步 | 150 步，≤768 token | Diamond 198 题，≤2048 token |
| `a100` | **4B 正式实验**（A100 40GB，约 3 小时） | 2 万 Dolci + 2000 格式样本，1 epoch | 150 步，≤1536 token | Diamond 198 + Main 448 题，≤4096 token |

⚠️ 快速评测**不是**排行榜分数（单样本、选项顺序与 lm-eval 不同），只用来比较同一次运行里各阶段的变化。
正式分数：把合并后的模型推到 HF，再在新的 runtime 里用 `main` 分支的 `01_eval_gpqa.ipynb` 评测。

**五次实验计划**（Qwen3-4B-Base，`MODE="a100"`；详见 `experiments/README.md`）：

| # | 改什么 | 要回答的问题 |
|---|---|---|
| X1 | Instruct（科学/数学）2 万 + SciQ 2000 → GRPO SciQ | 答完会不会停；4B 基线 |
| X2 | `THINK_N=8000`（短思维链）+ Instruct 8000 + SciQ 2000 | 短思维链有没有帮助 |
| X3 | `RUN_SFT=False`、`INIT_MODEL`=X1/X2 中更好的 SFT → `GRPO_DATA="mmlu_pro_stem"` | 更难的 RL 数据能不能提分 |
| X4 | 按 X3 调一项（`GRPO_LR`、步数或组大小） | GRPO 调参 |
| X5 | 最佳配置 → 合并 → 推 HF → `01` 锁定评测（4 seeds） | 正式分数 |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆提交分支（数据准备 / 去污染脚本在里面）
import os
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval
!git pull -q

In [ ]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
# 让 uv 一起解析 unsloth + vllm 的兼容版本（2026-11 实测可用：unsloth 2026.11.3、vllm 0.31–0.33、trl 1.15、transformers 5.17）。
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm wandb {_numpy} {_pil}

In [ ]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # Unsloth：GRPO 时让 vLLM 和训练共享显存
# T4（计算能力 7.5）不支持 FlashInfer 采样器；Unsloth 会给 vLLM 打开它，新版 vLLM 随即报错。这里关掉（A100 不受影响）。
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

In [ ]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需；先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（可选；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login
login(token = os.environ.get("HF_TOKEN"))
print("W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

## 0. 配置

In [ ]:
EXP_NAME     = "X1"                           # 实验编号：W&B run 名、报告里用
MODE         = "a100"                         # "quick" | "medium" | "a100"
MODEL_NAME   = "unsloth/Qwen3-4B-Base"        # quick 可用 "unsloth/Qwen3-0.6B-Base"，medium 可用 "unsloth/Qwen3-1.7B-Base"
BASE_MODEL   = "Qwen/Qwen3-4B-Base"           # 提交时填的原始 base（与 MODEL_NAME 对应）
LOAD_IN_4BIT = False                          # 16-bit LoRA（Unsloth 官方 GRPO 示例也是 False）

# 流程开关
RUN_SFT       = True
INIT_MODEL    = None    # 例如 "YOUR_HF_USERNAME/qwen3-4b-x1-sft"：从已经 SFT 并合并好的模型开始（配合 RUN_SFT=False，省掉 SFT 时间）
PUSH_SFT_REPO = None    # 例如 "YOUR_HF_USERNAME/qwen3-4b-x1-sft"：SFT 后合并并推到 HF 私有仓库，后面的实验直接复用
RUN_GRPO      = True

# SFT 数据：--include / --exclude 同时匹配 Dolci 的 source_dataset 和 domain（不区分大小写，先看 exclude）
DOLCI_DATASET    = "allenai/Dolci-Instruct-SFT-7B"
DOLCI_INCLUDE    = ["science", "math", "logic", "gsm"]                       # run2 里 OpenThoughts3+ Science、Persona MATH/GSM、Logic Puzzles…
DOLCI_EXCLUDE    = ["safety", "wildguard", "python", "code", "aya", "precise", "flan"]
THINK_DATASET    = "allenai/Dolci-Think-SFT-7B"
THINK_N          = 0                          # >0：再从 Dolci-Think 抽这么多条短思维链（X2 用 8000）
THINK_MAX_TOKENS = 3072                       # 只要 ≤ 这个长度的思维链样本，让模型学会在评测预算内推理完
THINK_INCLUDE    = ["science", "math"]

# GRPO
GRPO_DATA = "sciq"      # "sciq"（简单：主要练格式和停止）| "mmlu_pro_stem"（更难，更接近 GPQA）
GRPO_LR   = 5e-6

# W&B（照 rlhf-book code/policy_gradients/train.py：环境变量优先；没有 key 就关闭）
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")    # None = 你的默认账号

PRESETS = dict(
    quick = dict(max_seq_len=2048, gpu_util=0.7, lora_rank=32,
                 sft_dolci_n=200, sft_mcq_n=200, sft_max_tokens=1024, sft_steps=40, sft_lr=2e-4, sft_batch=4, sft_accum=2,
                 grpo_prompts=200, grpo_steps=30, grpo_max_completion=512, num_gen=8, grpo_accum=2,
                 eval_configs=["gpqa_diamond"], eval_n=60, eval_max_tokens=1024, shuffle_buffer=1000),
    medium = dict(max_seq_len=4096, gpu_util=0.7, lora_rank=32,
                  sft_dolci_n=1000, sft_mcq_n=500, sft_max_tokens=2048, sft_steps=150, sft_lr=2e-4, sft_batch=4, sft_accum=2,
                  grpo_prompts=1200, grpo_steps=150, grpo_max_completion=768, num_gen=8, grpo_accum=2,
                  eval_configs=["gpqa_diamond"], eval_n=None, eval_max_tokens=2048, shuffle_buffer=5000),
    a100 = dict(max_seq_len=6144, gpu_util=0.8, lora_rank=64,
                sft_dolci_n=20000, sft_mcq_n=2000, sft_max_tokens=3072, sft_steps=-1, sft_lr=1e-4, sft_batch=8, sft_accum=2,
                grpo_prompts=4000, grpo_steps=150, grpo_max_completion=1536, num_gen=8, grpo_accum=2,
                eval_configs=["gpqa_diamond", "gpqa_main"], eval_n=None, eval_max_tokens=4096, shuffle_buffer=20000),
)
Q = PRESETS[MODE]
MAX_SEQ_LEN, GPU_UTIL, LORA_RANK = Q["max_seq_len"], Q["gpu_util"], Q["lora_rank"]
print(EXP_NAME, MODE, MODEL_NAME, Q)

## 1. 加载模型（`fast_inference=True` = 训练和生成共用一个 vLLM 引擎）

In [ ]:
import os, torch
if torch.cuda.get_device_capability()[0] < 8:   # 再设一次，防止跳过了前面的格
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = INIT_MODEL or MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = LOAD_IN_4BIT,
    fast_inference = True,
    max_lora_rank = LORA_RANK,
    gpu_memory_utilization = GPU_UTIL,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
if tokenizer.chat_template is None:     # unsloth/Qwen3-*-Base 的 tokenizer 没有模板（run1 实测）
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")

# 停止：Base tokenizer 的 eos 是 <|endoftext|>，ChatML 每轮以 <|im_end|> 结束。run1/run2 里模型答完都没停
# （截断率 95–100%）。所以 eos 改成 <|im_end|>，并且采样时遇到 <|im_end|>、<|endoftext|>、<|im_start|>（开始新一轮）都停。
IM_END    = tokenizer.convert_tokens_to_ids("<|im_end|>")
ENDOFTEXT = tokenizer.convert_tokens_to_ids("<|endoftext|>")
IM_START  = tokenizer.convert_tokens_to_ids("<|im_start|>")
STOP_IDS  = [IM_END, ENDOFTEXT, IM_START]
STOP_STRS = ["<|im_end|>", "<|im_start|>", "<|endoftext|>"]   # 万一这些标记被当成普通文本写出来
tokenizer.eos_token = "<|im_end|>"
model.generation_config.eos_token_id = [IM_END, ENDOFTEXT]
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.pad_token = "<|endoftext|>"
_demo = tokenizer.apply_chat_template([{"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello"}], tokenize = False)
print(repr(_demo))
assert _demo.rstrip().endswith("<|im_end|>"), "chat template 应以 <|im_end|> 结束每轮"
print("eos:", tokenizer.eos_token, "| stop ids:", STOP_IDS, "| pad:", tokenizer.pad_token)

In [ ]:
# Weights & Biases（照 rlhf-book code/policy_gradients/train.py：没有 key 或 project 就 mode="disabled"）
import wandb
USE_WANDB = bool(os.environ.get("WANDB_API_KEY")) and bool(WANDB_PROJECT)
RUN_NAME = f"{EXP_NAME}-{MODEL_NAME.split('/')[-1]}-{MODE}"
wandb.init(
    project = WANDB_PROJECT, entity = WANDB_ENTITY, name = RUN_NAME,
    mode = "online" if USE_WANDB else "disabled",
    config = dict(exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL, base_model = BASE_MODEL,
                  load_in_4bit = LOAD_IN_4BIT, run_sft = RUN_SFT, run_grpo = RUN_GRPO,
                  dolci = DOLCI_DATASET, dolci_include = DOLCI_INCLUDE, dolci_exclude = DOLCI_EXCLUDE,
                  think_n = THINK_N, think_max_tokens = THINK_MAX_TOKENS, grpo_data = GRPO_DATA, grpo_lr = GRPO_LR,
                  gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS, **Q),
)
REPORT_TO = "wandb" if USE_WANDB else "none"
print("W&B:", wandb.run.url if USE_WANDB else "disabled")

## 2. GPQA 快速评测（同一个 vLLM 引擎，只输出聚合数字；不记录题目或模型输出）

题面与 lm-eval 的 `gpqa_*_cot_zeroshot` 模板一致。`strict` = 出现 `The answer is (X)`；`flex` = 输出中最后一个 `(A)`–`(D)`（排行榜用的是 flex）。

In [ ]:
import random, re, json, statistics, time
from collections import Counter
from datasets import load_dataset
from vllm import SamplingParams

TEMPLATE = ("What is the correct answer to this question:{q}\nChoices:\n"
            "(A) {a}\n(B) {b}\n(C) {c}\n(D) {d}\nLet's think step by step: ")
STRICT_RE = re.compile(r"The answer is \(([A-D])\)")
LETTER_RE = re.compile(r"\(([A-D])\)")

def _clean(t):
    return " ".join(str(t).replace(" [title]", ". ").split())

def build_gpqa(config, n = None):
    ds = load_dataset("Idavidrein/gpqa", config, split = "train")
    rows = []
    for i, d in enumerate(ds):
        choices = [_clean(d["Incorrect Answer 1"]), _clean(d["Incorrect Answer 2"]),
                   _clean(d["Incorrect Answer 3"]), _clean(d["Correct Answer"])]
        correct = choices[3]
        random.Random(1000 + i).shuffle(choices)
        rows.append((TEMPLATE.format(q = _clean(d["Question"]), a = choices[0], b = choices[1],
                                     c = choices[2], d = choices[3]), "ABCD"[choices.index(correct)]))
    return rows[:n] if n else rows

def _chat(prompt):
    return tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize = False, add_generation_prompt = True)

EVAL_SETS = {}
for cfg in Q["eval_configs"]:
    rows = build_gpqa(cfg, Q["eval_n"])
    texts = [_chat(p) for p, _ in rows]
    longest = max(len(tokenizer(t, add_special_tokens = False)["input_ids"]) for t in texts)
    EVAL_SETS[cfg] = (rows, texts, longest)
EVAL_MAX_TOKENS = min(Q["eval_max_tokens"], MAX_SEQ_LEN - max(v[2] for v in EVAL_SETS.values()) - 8)
if EVAL_MAX_TOKENS < Q["eval_max_tokens"]:
    print(f"[!] eval_max_tokens clamped to {EVAL_MAX_TOKENS} to fit MAX_SEQ_LEN={MAX_SEQ_LEN}")
RESULTS = {}

def quick_eval(tag, lora_request = None):
    sp = SamplingParams(temperature = 0.6, top_p = 0.95, max_tokens = EVAL_MAX_TOKENS, seed = 0,
                        stop_token_ids = STOP_IDS, stop = STOP_STRS)
    RESULTS[tag] = {}
    for cfg, (rows, texts, _) in EVAL_SETS.items():
        t0 = time.time()
        outs = model.fast_generate(texts, sampling_params = sp, lora_request = lora_request)
        strict = flex = extracted = truncated = 0
        lengths, letters = [], Counter()
        for o, (_, gold) in zip(outs, rows):
            c = o.outputs[0]
            m, f = STRICT_RE.findall(c.text), LETTER_RE.findall(c.text)
            strict += bool(m) and m[0] == gold          # lm-eval strict-match 取第一个
            flex += bool(f) and f[-1] == gold           # lm-eval flexible-extract 取最后一个
            extracted += bool(f)
            truncated += c.finish_reason == "length"
            lengths.append(len(c.token_ids))
            letters[f[-1] if f else "none"] += 1
        n = len(rows)
        r = dict(n = n, acc_flex = round(100 * flex / n, 1), acc_strict = round(100 * strict / n, 1),
                 extract_rate = round(100 * extracted / n, 1), truncation_rate = round(100 * truncated / n, 1),
                 mean_tokens = round(statistics.fmean(lengths)), letters = dict(sorted(letters.items())),
                 minutes = round((time.time() - t0) / 60, 1))
        RESULTS[tag][cfg] = r
        for k, v in r.items():
            if isinstance(v, (int, float)):
                wandb.run.summary[f"gpqa/{cfg}/{tag}/{k}"] = v
        print(tag, cfg, r)

def diagnose(tag, lora_request = None, k = 3):
    """看清楚答案后面生成了什么（SciQ 验证集，公开数据；不跳过特殊 token）。"""
    val = load_dataset("allenai/sciq", split = "validation").select(range(k))
    prompts = [_chat(sciq_mcq(ex, 10_000 + i)[0]) for i, ex in enumerate(val)]
    sp = SamplingParams(temperature = 0.0, max_tokens = 512, stop_token_ids = STOP_IDS, stop = STOP_STRS,
                        skip_special_tokens = False)
    for o in model.fast_generate(prompts, sampling_params = sp, lora_request = lora_request):
        c = o.outputs[0]
        print(f"--- [{tag}] finish={c.finish_reason} stop={c.stop_reason!r} tokens={len(c.token_ids)}")
        print(repr(c.text[-400:]))

quick_eval("0_init" if INIT_MODEL else "0_base")

## 3. SFT（照 Unsloth Instruct notebook：chat 模板转成 `text` + `train_on_responses_only`）

1. **Dolci-Instruct**（OLMo 3 SFT 数据）按来源 / domain 过滤到科学、数学、逻辑，长度 ≤ `sft_max_tokens`。
2. 可选 **Dolci-Think** 短思维链（`THINK_N > 0`）。
3. **选择题格式样本**：SciQ 训练集套 GPQA 模板，答案 = `support` 段落 + `The answer is (X).`。

全部过一遍 GPQA 13-gram 去污染。

In [ ]:
import os
os.makedirs("data", exist_ok = True)
N_DOLCI, SFT_MAX, SHUF = Q["sft_dolci_n"], Q["sft_max_tokens"], Q["shuffle_buffer"]
inc = " ".join(DOLCI_INCLUDE); exc = " ".join(DOLCI_EXCLUDE)
inc_arg = f"--include {inc}" if inc else ""
if RUN_SFT:
    !python scripts/prepare_dolci_sft.py --dataset {DOLCI_DATASET} --out data/dolci.jsonl --n {N_DOLCI} --tokenizer {MODEL_NAME} --max-tokens {SFT_MAX} --shuffle-buffer {SHUF} {inc_arg} --exclude {exc}
    if THINK_N > 0:
        t_inc = " ".join(THINK_INCLUDE)
        !python scripts/prepare_dolci_sft.py --dataset {THINK_DATASET} --out data/think.jsonl --n {THINK_N} --tokenizer {MODEL_NAME} --max-tokens {THINK_MAX_TOKENS} --shuffle-buffer {SHUF} --include {t_inc} --exclude {exc}

In [ ]:
SCIQ = load_dataset("allenai/sciq", split = "train").shuffle(seed = 0)

def sciq_mcq(ex, idx):
    rng = random.Random(idx)
    choices = [ex["distractor1"], ex["distractor2"], ex["distractor3"], ex["correct_answer"]]
    rng.shuffle(choices)
    letter = "ABCD"[choices.index(ex["correct_answer"])]
    prompt = TEMPLATE.format(q = ex["question"], a = choices[0], b = choices[1], c = choices[2], d = choices[3])
    return prompt, letter

# SciQ 前一段给 SFT，后一段给 GRPO，互不重叠
sft_mcq, i = [], 0
for i, ex in enumerate(SCIQ):
    if len(sft_mcq) >= Q["sft_mcq_n"]:
        break
    if ex["support"].strip():
        prompt, letter = sciq_mcq(ex, i)
        sft_mcq.append({"messages": [{"role": "user", "content": prompt},
                                     {"role": "assistant", "content": ex["support"].strip() + f"\n\nThe answer is ({letter})."}]})
SCIQ_GRPO_START = i

DATA = {}
if RUN_SFT:
    def _load(path):
        return [json.loads(l) for l in open(path)] if os.path.exists(path) else []
    dolci, think = _load("data/dolci.jsonl"), (_load("data/think.jsonl") if THINK_N > 0 else [])
    sft_rows = [{"messages": r["prompt"] + r["completion"]} for r in dolci + think] + sft_mcq
    with open("data/sft_all.jsonl", "w") as f:
        for r in sft_rows:
            f.write(json.dumps(r, ensure_ascii = False) + "\n")
    !python scripts/decontam.py --train-jsonl data/sft_all.jsonl --out data/decontam_sft.json > /dev/null
    bad = set(json.load(open("data/decontam_sft.json"))["flagged_row_indices"])
    sft_rows = [r for j, r in enumerate(sft_rows) if j not in bad]
    random.Random(0).shuffle(sft_rows)
    DATA = dict(dolci = len(dolci), think = len(think), sciq_format = len(sft_mcq), dropped_decontam = len(bad),
                dolci_tokens = sum(r["n_tokens"] for r in dolci), think_tokens = sum(r["n_tokens"] for r in think),
                dolci_sources = Counter(r["source"] for r in dolci).most_common(10))
    wandb.config.update({"data": DATA}, allow_val_change = True)
    print(DATA)

In [ ]:
SFT = {}
if RUN_SFT:
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth.chat_templates import train_on_responses_only

    # 以 <|im_end|>（现在的 eos）结尾，避免 TRL 再额外追加一个 eos
    sft_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize = False).rstrip("\n")}
                                for r in sft_rows])
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = sft_ds,
        args = SFTConfig(
            dataset_text_field = "text",
            per_device_train_batch_size = Q["sft_batch"],
            gradient_accumulation_steps = Q["sft_accum"],
            warmup_steps = 10,
            max_steps = Q["sft_steps"],
            num_train_epochs = 1,
            learning_rate = Q["sft_lr"],
            logging_steps = 5,
            optim = "adamw_8bit",
            weight_decay = 0.001,
            lr_scheduler_type = "linear",
            seed = 3407,
            report_to = REPORT_TO,
            run_name = RUN_NAME + "-sft",
            output_dir = "outputs/sft",
            save_strategy = "no",
        ),
    )
    trainer = train_on_responses_only(trainer, instruction_part = "<|im_start|>user\n",
                                      response_part = "<|im_start|>assistant\n")
    # 诊断：训练标签的最后几个 token 里应该有 <|im_end|>（否则模型学不会停）
    try:
        _labels = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
        print("label tail:", repr(tokenizer.decode(_labels[-12:])), "| contains <|im_end|>:", IM_END in _labels)
    except Exception as e:
        print("label check skipped:", e)

    sft_stats = trainer.train()
    sft_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    SFT = dict(steps = trainer.state.global_step, minutes = round(sft_stats.metrics["train_runtime"] / 60, 1),
               loss_first = sft_losses[0] if sft_losses else None, loss_last = sft_losses[-1] if sft_losses else None)
    print(SFT)
    model.save_lora("sft_lora")
    quick_eval("1_sft", model.load_lora("sft_lora"))
    diagnose("1_sft", model.load_lora("sft_lora"))
    if PUSH_SFT_REPO:
        try:
            model.push_to_hub_merged(PUSH_SFT_REPO, tokenizer, save_method = "merged_16bit",
                                     private = True, token = os.environ.get("HF_TOKEN"))
        except TypeError:   # 旧版 Unsloth 没有 private 参数：先在 HF 网页上建好私有仓库再推
            model.push_to_hub_merged(PUSH_SFT_REPO, tokenizer, save_method = "merged_16bit",
                                     token = os.environ.get("HF_TOKEN"))
        print("SFT pushed to", PUSH_SFT_REPO, "→ 后面的实验设 INIT_MODEL 为它、RUN_SFT=False")
    del sft_ds, trainer
    torch.cuda.empty_cache(); import gc; gc.collect()

## 4. GRPO（照 Unsloth GRPO notebook：`vllm_sampling_params` + 多个奖励函数）

题库（`GRPO_DATA`）：`sciq`（SciQ 中没用于 SFT 的部分）或 `mmlu_pro_stem`（MMLU-Pro 的物理/化学/生物/工程/数学，抽成 4 选 1）。题面都套 GPQA 模板。奖励：
- `format_reward`：`The answer is (X)` 恰好出现一次 → +1.0
- `correctness_reward`：最后一个 `(A)`–`(D)` 正确 → +2.0；完全没有选项字母 → −1.0
- `stop_reward`：回答打满 `max_completion_length`（没停）→ −1.0（run2 里 GRPO 的回答几乎全是满长度）

In [ ]:
import numpy as np
from datasets import Dataset

def mmlu_pro_stem_rows(n):
    ds = load_dataset("TIGER-Lab/MMLU-Pro", split = "test").shuffle(seed = 0)
    keep = {"physics", "chemistry", "biology", "engineering", "math"}
    rows = []
    for idx, ex in enumerate(ds):
        if len(rows) >= n:
            break
        if ex["category"] not in keep:
            continue
        opts = [o for o in ex["options"] if str(o).strip() and str(o).strip() != "N/A"]
        correct = ex["options"][ex["answer_index"]]
        others = [o for o in opts if o != correct]
        if len(others) < 3:
            continue
        rng = random.Random(idx)
        choices = rng.sample(others, 3) + [correct]
        rng.shuffle(choices)
        prompt = TEMPLATE.format(q = ex["question"], a = choices[0], b = choices[1], c = choices[2], d = choices[3])
        rows.append({"prompt": [{"role": "user", "content": prompt}], "answer": "ABCD"[choices.index(correct)]})
    return rows

if RUN_GRPO:
    if GRPO_DATA == "mmlu_pro_stem":
        grpo_rows = mmlu_pro_stem_rows(Q["grpo_prompts"])
    else:
        grpo_rows = []
        for j in range(SCIQ_GRPO_START, len(SCIQ)):
            if len(grpo_rows) >= Q["grpo_prompts"]:
                break
            prompt, letter = sciq_mcq(SCIQ[j], j)
            grpo_rows.append({"prompt": [{"role": "user", "content": prompt}], "answer": letter})
    # 去污染（GRPO 题目同样不能和 GPQA 重叠）
    with open("data/grpo_prompts.jsonl", "w") as f:
        for r in grpo_rows:
            f.write(json.dumps({"prompt": r["prompt"]}, ensure_ascii = False) + "\n")
    !python scripts/decontam.py --train-jsonl data/grpo_prompts.jsonl --out data/decontam_grpo.json > /dev/null
    bad = set(json.load(open("data/decontam_grpo.json"))["flagged_row_indices"])
    grpo_rows = [r for j, r in enumerate(grpo_rows) if j not in bad]

    # 当前 TRL 不支持 max_prompt_length（Unsloth 会提示忽略），像 Unsloth 示例那样过滤掉最长的 10% 题目
    def _n_tokens(msgs):
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt = True, tokenize = True)
        return len(ids["input_ids"] if hasattr(ids, "keys") else ids)
    lens = [_n_tokens(r["prompt"]) for r in grpo_rows]
    max_prompt_length = int(np.quantile(lens, 0.9)) + 1
    grpo_ds = Dataset.from_list(grpo_rows).select(np.where(np.array(lens) < max_prompt_length)[0])
    max_completion_length = min(Q["grpo_max_completion"], MAX_SEQ_LEN - max_prompt_length)
    GRPO_INFO = dict(data = GRPO_DATA, prompts = len(grpo_ds), dropped_decontam = len(bad),
                     max_prompt_length = max_prompt_length, max_completion_length = max_completion_length)
    wandb.config.update({"grpo_info": GRPO_INFO}, allow_val_change = True)
    print(GRPO_INFO)

def format_reward(completions, **kwargs):
    return [1.0 if len(STRICT_RE.findall(c[0]["content"])) == 1 else 0.0 for c in completions]

def correctness_reward(completions, answer, **kwargs):
    out = []
    for c, a in zip(completions, answer):
        found = LETTER_RE.findall(c[0]["content"])
        out.append(-1.0 if not found else (2.0 if found[-1] == a else 0.0))
    return out

def stop_reward(completions, completion_ids = None, **kwargs):
    if completion_ids is None:
        completion_ids = [tokenizer(c[0]["content"], add_special_tokens = False)["input_ids"] for c in completions]
    return [-1.0 if len(ids) >= max_completion_length else 0.0 for ids in completion_ids]

In [ ]:
GRPO = {}
if RUN_GRPO:
    from trl import GRPOConfig, GRPOTrainer
    vllm_sampling_params = SamplingParams(
        min_p = 0.1, top_p = 1.0, top_k = -1, seed = 3407,
        stop_token_ids = STOP_IDS, stop = STOP_STRS,
    )
    NUM_GEN = Q["num_gen"]
    training_args = GRPOConfig(
        vllm_sampling_params = vllm_sampling_params,
        generation_kwargs = {"stop_token_ids": STOP_IDS},
        temperature = 1.0,
        learning_rate = GRPO_LR,
        weight_decay = 0.001,
        warmup_steps = max(1, Q["grpo_steps"] // 10),
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        logging_steps = 1,
        per_device_train_batch_size = NUM_GEN,          # 每步 = grpo_accum 道题 × NUM_GEN 个采样
        gradient_accumulation_steps = Q["grpo_accum"],
        num_generations = NUM_GEN,
        max_completion_length = max_completion_length,
        # loss_type / beta 不设：Unsloth 当前默认 loss_type="dapo", beta=0.0（无 KL）
        max_steps = Q["grpo_steps"],
        save_steps = 10_000,
        log_completions = False,                         # 不往 W&B 上传生成内容
        report_to = REPORT_TO,
        run_name = RUN_NAME + "-grpo",
        output_dir = "outputs/grpo",
    )
    trainer = GRPOTrainer(
        model = model,
        processing_class = tokenizer,
        reward_funcs = [format_reward, correctness_reward, stop_reward],
        args = training_args,
        train_dataset = grpo_ds,
    )
    grpo_stats = trainer.train()

    def _avg(key, part):
        vals = [h[key] for h in trainer.state.log_history if key in h]
        k = max(1, len(vals) // 5)
        return round(float(np.mean(vals[:k] if part == "first" else vals[-k:])), 3) if vals else None

    GRPO = dict(steps = trainer.state.global_step, minutes = round(grpo_stats.metrics["train_runtime"] / 60, 1), **GRPO_INFO)
    for key in ["reward", "rewards/format_reward/mean", "rewards/correctness_reward/mean", "rewards/stop_reward/mean",
                "completions/mean_length", "completions/clipped_ratio", "frac_reward_zero_std", "completion_length"]:
        if any(key in h for h in trainer.state.log_history):
            GRPO[key] = {"first": _avg(key, "first"), "last": _avg(key, "last")}
    print(GRPO)
    model.save_lora("grpo_lora")
    quick_eval("2_grpo", model.load_lora("grpo_lora"))
    diagnose("2_grpo", model.load_lora("grpo_lora"))

## 5. 汇总（W&B 上也有同样的表；最后打印的 JSON 可以贴回给 Claude）

In [ ]:
cols = ["stage", "set", "acc_flex", "acc_strict", "extract_rate", "truncation_rate", "mean_tokens", "minutes"]
print(" ".join(f"{c:>15s}" for c in cols))
table = wandb.Table(columns = cols)
for tag, per_set in RESULTS.items():
    for cfg, v in per_set.items():
        row = [tag, cfg] + [v[c] for c in cols[2:]]
        table.add_data(*row)
        print(" ".join(f"{str(x):>15s}" for x in row))
print("SFT:", SFT); print("GRPO:", GRPO)
REPORT = dict(notebook = "11_sft_grpo_unsloth", exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL,
              gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, q = Q, dolci = DOLCI_DATASET, dolci_include = DOLCI_INCLUDE,
              think_n = THINK_N, grpo_data = GRPO_DATA, grpo_lr = GRPO_LR, data = DATA, sft = SFT, grpo = GRPO,
              quick_eval = RESULTS, versions = VERSIONS, wandb = wandb.run.url if USE_WANDB else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1)
if USE_WANDB:
    wandb.log({"gpqa_quick_eval": table})
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False))

## 6.（可选）合并保存 → 推到 Hub → 用 `main` 分支的 `01` 做正式评测

`merged_16bit` 把 LoRA 合并进权重并保存（排行榜要求）；保存的 tokenizer 带着 eos=`<|im_end|>`，lm-eval 评测时才会停。
推到 Hub 后，**换一个新的 runtime** 打开 [`01_eval_gpqa.ipynb`](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/01_eval_gpqa.ipynb)，
`MODEL` 填你的仓库名、`BASE` 填上面的 `BASE_MODEL`。在 A100 上跑出来的模型提交时请在 `notes` 里注明 A100（不参与排名）。

In [ ]:
PUSH = False
HUB_REPO = "YOUR_HF_USERNAME/" + MODEL_NAME.split("/")[-1].lower() + f"-{EXP_NAME.lower()}"
if PUSH:
    model.push_to_hub_merged(HUB_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
    print(f"pushed: https://huggingface.co/{HUB_REPO}")
else:
    model.save_pretrained_merged("outputs/merged_16bit", tokenizer, save_method = "merged_16bit")
    print("saved locally: outputs/merged_16bit")